# Per-gene dose-response curve comparison

Generalized, dataset-agnostic version of `GEX_comp_Doming_Morris.ipynb` -- compares fitted bayesDREAM models across all overlapping trans genes, for however many datasets have a completed fit for a given cis gene. Panel shape auto-adapts: 2 datasets -> 2x2, 3 datasets -> 2x3.

Each panel:
- row 0: every dataset standalone (its own data + its own curve, with fitted-parameter markers)
- row 1: every dataset's own data + own curve again, with every *other* available dataset's curve overlaid on top

plus one guide-density panel (log2FC(x_true) by guide/cell_line) shared across all genes for a given cis gene.

**Prerequisite:** for each (dataset, cis_gene) you want here, `save_model_for_plotting()` (see `save_for_plotting.py` at the repo root) must already have been run once in the original fitting session, with its output directory registered in `comparative/datasets.py`'s `DatasetSpec.save_for_plotting_dir_fn`. This is a full model reload, so it's only meant for a bounded number of genes (Domingo's ~91 shared trans genes is fine; don't point this at Morris/Replogle transcriptome-wide -- use `trans_param_comparison.ipynb` for that instead).

As of this writing, `save_model_for_plotting()` has only actually been run for **Domingo GFI1B** and **Morris GFI1B** (the original ad hoc comparison). To get panels for NFE2/MYB/TET2 (and Morris/Replogle more generally), run it once per (dataset, cis_gene) pair first -- the automation below will just skip (with a printed message) any cis gene where a required export is still missing, rather than crash.

In [ ]:
# run "pip install ipython-autotime" in your conda env
%load_ext autotime

import os
import sys

# Derive the repo root from bayesDREAM's installed location (pip -e .), not
# from os.getcwd() -- the notebook's cwd at kernel start isn't guaranteed to
# be its own directory (depends on how Jupyter/the IDE was launched), so a
# '../..'-from-cwd guess silently fails to find comparative/ in that case.
# importlib.import_module (not a plain 'import bayesDREAM as ...') deliberately
# sidesteps a name collision: the bayesDREAM PACKAGE and the bayesDREAM CLASS it
# exports both share the literal name 'bayesDREAM'. If this cell's import ever
# gets merged with a 'from bayesDREAM import bayesDREAM' line elsewhere in the
# notebook, a plain import here can end up aliasing the class (no __file__)
# instead of the package -- this form can't be shadowed that way.
import importlib
_bayesdream_pkg = importlib.import_module('bayesDREAM')
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(_bayesdream_pkg.__file__)))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
import matplotlib.pyplot as plt

from comparative.datasets import DOMINGO, MORRIS, REPLOGLE
from comparative.dose_response_panels import (
    compare_datasets,
    compare_all_domingo_cis_genes,
    load_model_for_plotting,
    make_panel,
    resolve_sum_factor_col,
    allsig_copy,
)

## Config

In [ ]:
deviceno = 2
DEVICE = f'cuda:{deviceno}' if torch.cuda.is_available() else 'cpu'

PLOT_DIR = './dose_response_comparison_plots'

# Standalone panels (row 0) show fitted-parameter markers (EC50/inflection
# lines) by default. Turn off if they read as confusing next to the
# dataset-colour curves -- overlay panels (row 1) never show markers
# regardless of this flag.
SHOW_PARAM_MARKERS = True

## Automated: every Domingo cis gene

The main entry point. Loops over every cis gene in `DOMINGO.cis_genes` (GFI1B, NFE2, MYB, TET2 -- Domingo bounds the trans gene panel size, so it's the dataset this loop is driven by), and for each one automatically uses whichever of {Domingo, Morris, Replogle} actually has a completed fit for that gene (checked via each `DatasetSpec.cis_genes`):

- **GFI1B, NFE2**: all 3 datasets have it -> 2x3 panel per trans gene
- **MYB, TET2**: Morris never fit these (see `publication_runs/morris/config.yaml`'s `primary_genes`) -> 2x2 Domingo-vs-Replogle panel per trans gene

No manual per-gene setup needed -- just run this cell. Writes into `PLOT_DIR/<cis_gene>/`.

In [ ]:
results = compare_all_domingo_cis_genes(
    out_dir=PLOT_DIR,
    datasets=[DOMINGO, MORRIS, REPLOGLE],
    show_param_markers=SHOW_PARAM_MARKERS,
    device=DEVICE,
)
print('\nSummary:')
for cis_gene, genes in results.items():
    print(f'  {cis_gene}: {len(genes)} trans genes plotted')

## One cis gene at a time

Useful while iterating on plot styling, or to force a specific dataset subset instead of auto-detecting from `cis_genes`.

In [ ]:
CIS_GENE = 'GFI1B'
SPECS = [DOMINGO, MORRIS, REPLOGLE]  # or e.g. [DOMINGO, REPLOGLE] to force just two

plotted = compare_datasets(
    SPECS, CIS_GENE,
    out_dir=os.path.join(PLOT_DIR, CIS_GENE),
    show_param_markers=SHOW_PARAM_MARKERS,
    device=DEVICE,
)
print(f'Plotted {len(plotted)} genes: {plotted[:10]}{"..." if len(plotted) > 10 else ""}')

## Inspect a single panel inline

Reload the models for `CIS_GENE` once, then call `make_panel` directly for one gene at a time -- faster than re-running the whole loop above while tweaking plot styling.

In [ ]:
models = [load_model_for_plotting(s, CIS_GENE, device=DEVICE) for s in SPECS]
sfcols = [resolve_sum_factor_col(s, m) for s, m in zip(SPECS, models)]
summaries = [m.save_trans_summary(compute_lfc_ci=False, compute_derivative_roots=False) for m in models]
summaries_allsig = [allsig_copy(s) for s in summaries]

In [ ]:
GOI = plotted[0] if plotted else None
fig, unified_x = make_panel(
    GOI, SPECS, models, summaries_allsig, sfcols,
    cis_gene=CIS_GENE, show_param_markers=SHOW_PARAM_MARKERS,
)
plt.show()

## Customising further

- `compare_all_domingo_cis_genes(bounding_dataset=REPLOGLE)` would instead loop over Replogle's (larger) cis-gene list -- not recommended for genome-wide trans sets, but fine if you've cherry-picked a short `genes=[...]` list.
- Pass `genes=[...]` to `compare_datasets`/`compare_all_domingo_cis_genes` to restrict to a short hand-picked trans gene list instead of every shared one (useful for a quick look, or when a dataset's full trans panel is too big for this per-gene-reload approach).
- Each `DatasetSpec` in `comparative/datasets.py` carries its own fit-curve `.color` and `.cell_line_palette` -- edit there, not here, to keep both notebooks in sync.
- `make_panel`'s row-1 "own curve" is force-rendered regardless of that dataset's own FDR significance (matching the original 2-way panels' intent: show the shape even where it isn't a formal call); row-0's standalone curve uses real per-dataset significance gating, so a blank row-0 curve for a given dataset is real signal, not a bug.